# Notebook07 — FMM-inspired energy encoder pilot

**Run All starts the real detached experiment and returns while training continues.** The experiment has a hard two-hour limit from first launch, including calibration, exact-resume checks, training and evaluation. Expected runtime: roughly 90–120 minutes, finalized from actual-update timing. No results are claimed before completion.

**Question:** Does the mathematically defined convex energy encoder learn better Food101 representations per unit of measured compute than our ViT encoder?

**Design:** Two arms, ViT and energy encoder; prefer three paired seeds. Same image/mask/augmentation exposure, shared patch stem and predictor initialization, width192, patch8, FP32 and effective batch128. Architecture and parameter counts differ and are reported. Energy uses four steps with η=.5, λnear=λfar=.25, and far-summary width32. Its unique minimizer and contraction guarantee concern inference, not representation quality. The predictor retains its existing attention.

**Inputs:** Existing sealed 10,100-image Food101 training subset, 10,100 labeled probe-training images and 2,020 development images. Official test data stays reserved. Development data has been reused in prior research, so this is exploratory.

**Outputs:** `artifacts/notebook_07/energy-encoder-001/`; final initial-versus-trained probe quality, paired seed differences, learning gains, compute/memory and collapse diagnostics. Full-state checkpoints under `checkpoints/energy-encoder-001/`; retain latest two and best, plus final evaluation snapshots. Checkpoint every50 updates or15 minutes, at epoch/stage boundaries, with atomic manifests and one writer.

**Exact resume:** On the `ahr-go` kernel, rerun this notebook unchanged with the same RUN_ID before its original deadline. It reconnects to an existing tmux worker or resumes verified saved state. It never resets the clock or silently restarts invalid state. After the deadline, checkpoints remain available; continued training requires a separately authorized budget. The workspace is on persistent `/dev/sda1`; instance-deletion retention is unverified.


## 1. Load tagged definitions

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat, importlib.util, sys
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa").resolve()
_boot = {}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)
_spec = importlib.util.spec_from_loader("fmm07_workflow", _boot["TaggedNotebookLoader"](ROOT / "experiments/07_energy/workflow.ipynb"))
WORKFLOW = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = WORKFLOW
_spec.loader.exec_module(WORKFLOW)


## 2. Registered configuration

In [ ]:
RUN_ID = "energy-encoder-001"
CONFIG = WORKFLOW.study_configuration(RUN_ID)
CONFIG

## 3. Launch or reconnect
This starts the actual detached worker. Before paired training it calibrates timing and verifies exact checkpoint recovery. An independent watcher enforces the original deadline.

In [ ]:
LAUNCH = WORKFLOW.launch_study(ROOT, CONFIG)
LAUNCH

## 4. Inspect progress
Rerun this cell to refresh. Attach with `tmux attach -t fmm-energy-energy-encoder-001`; detach with Ctrl-b, then d. Results will appear at `artifacts/notebook_07/energy-encoder-001/stages/learning/stage_results.md`.

In [ ]:
STATUS = WORKFLOW.inspect_study(ROOT, RUN_ID)
STATUS

## Interpretation
Successful execution and energy descent establish implementation correctness. Useful learning requires improvement over each encoder’s own initialization; superiority requires paired comparisons against ViT, with uncertainty and actual compute costs. A confidence interval spanning zero remains unresolved. Small seed counts and reused development data limit the conclusions. A cutoff retains INCOMPLETE and resumable checkpoints. This pilot compares complete encoder architectures; it does not isolate the causal contribution of far interactions.